# EDA of `r/litigi`

In [ ]:
from subreddit_lens import load_comments
import pandas as pd
import plotly.express as px
from pathlib import Path

from subreddit_lens import load_config

# Locate the analytics directory, which holds subreddit-lens.toml, so paths
# work regardless of the working directory.
ANALYTICS_DIR = next(
    p
    for p in [Path.cwd(), Path.cwd() / "analytics", *Path.cwd().parents]
    if (p / "subreddit-lens.toml").exists()
)
config = load_config(ANALYTICS_DIR / "subreddit-lens.toml")
DATA_DIR = config.data_dir
OUTPUT_DIR = config.output_dir
OUTPUT_DIR.mkdir(exist_ok=True)
# r/litigi is an Italian subreddit: analyse hours in local time.
TZ = config.timezone

## Load and preprocess the data

In [ ]:
filename = DATA_DIR / "litigi_comments.parquet"
litigi = load_comments(filename)
litigi['created_dt'] = pd.to_datetime(litigi['created_utc'], unit='s', utc=True).dt.tz_convert(TZ)
litigi.head()

In [ ]:
litigi.describe()

In [ ]:
monthly_contrib = litigi.groupby(litigi['created_dt'].dt.tz_localize(None).dt.to_period("M")).agg(
    Count=('id', 'size')
).reset_index()

# Convert period to timestamp for proper plotting
monthly_contrib.rename(columns={'created_dt': 'Month'}, inplace=True)
monthly_contrib['Month'] = monthly_contrib['Month'].dt.to_timestamp()

# Create bar plot
fig = px.bar(monthly_contrib, x='Month', y='Count',
             labels={'Month': 'Month', 'Count': 'Number of Comments'},
             title="Monthly User Comments on r/litigi",
             color_discrete_sequence=["#1f77b4"],color='Count',color_continuous_scale='viridis')  # High-contrast blue

# Improve layout
fig.update_layout(
    xaxis=dict(
        tickformat="%b %Y",  # Show abbreviated month and year
        tickmode="auto",      # Auto-adjusts to fit the range
        showgrid=True
    ),
    yaxis=dict(showgrid=True),
    title_x=0.5  # Centers the title
)

fig.show()

In [ ]:
hour_activity = litigi.groupby(litigi['created_dt'].dt.hour).agg(
    Count=('id', 'size')
).reset_index()

# Rename column for clarity
hour_activity.rename(columns={'created_dt': 'Hour'}, inplace=True)
fig = px.bar(hour_activity, x='Hour', y='Count', 
             labels={'Hour': 'Hour of the Day', 'Count': 'Number of Comments'},
             title="Hourly Activity on r/litigi",
             color='Count',color_continuous_scale='viridis')
fig.update_layout(
    xaxis=dict(
        tickmode='linear',
        tickvals=list(range(24)),
    ),
    yaxis=dict(showgrid=True),
    title_x=0.5  # Centers title
)

fig.show()


In [ ]:
week_activity = litigi.groupby(litigi['created_dt'].dt.day_of_week).agg(
    Count=('id', 'size')
).reset_index()
days=['Lun','Mar','Mer','Gio','Ven','Sab','Dom']
week_activity['Weekday']=days
# Rename column for clarity
#week_activity.rename(columns={'created_dt': 'Weekday'}, inplace=True)
fig = px.bar(week_activity, x='Weekday', y='Count', 
             labels={'Weekday': 'Day of the week', 'Count': 'Number of Comments'},
             title="Weekly Activity on r/litigi",
             color='Count',color_continuous_scale='viridis')
fig.update_layout(
    xaxis=dict(
        tickmode='linear',
        tickvals=days
    ),
    yaxis=dict(showgrid=True),
    title_x=0.5  # Centers title
)

fig.show()

## Evolution of hourly posting

In [ ]:
import pandas as pd
import plotly.express as px

# Ensure 'created_dt' is a datetime column

# Extract relevant time features
litigi['year_month'] = litigi['created_dt'].dt.tz_localize(None).dt.to_period('M')  # Year-Month format
litigi['hour'] = litigi['created_dt'].dt.hour  # Hour of the day

# Count occurrences for each (year_month, hour) pair
heatmap_data = litigi.groupby(['year_month', 'hour']).size().reset_index(name='count')

# Convert 'year_month' to string for Plotly
heatmap_data['year_month'] = heatmap_data['year_month'].astype(str)

# Create the heatmap
fig = px.imshow(
    heatmap_data.pivot(columns='year_month', index='hour', values='count'),
    labels=dict(x="Month", y="Hour of Day", color="Post Count"),
    y=list(range(24)),  # Ensure x-axis is in correct order
    x=heatmap_data['year_month'].unique(),
    color_continuous_scale="Viridis"
)

fig.update_layout(
    title="User Commenting Habits Evolution (Hourly Distribution per Month)",
    xaxis_title="Month",
    yaxis_title="Hour of Day",
    yaxis=dict(autorange="reversed"),  # Hour 0 at the top
    coloraxis_colorbar=dict(title="Post Count")
)

fig.show()


In [ ]:
# Convert timestamp
litigi['created_dt'] = pd.to_datetime(litigi['created_utc'], unit='s', utc=True).dt.tz_convert(TZ)

# Extract features
litigi['month'] = litigi['created_dt'].dt.tz_localize(None).dt.to_period('M').astype(str)
litigi['hour'] = litigi['created_dt'].dt.hour

# Aggregate and normalize by month
post_counts = pd.crosstab(litigi['month'], litigi['hour'])
post_counts = post_counts.reindex(columns=range(24), fill_value=0)
post_counts = post_counts.sort_index()

# Normalize to get hourly percentages within each month
df_normalized = post_counts.div(post_counts.sum(axis=1), axis=0) * 100

# Transpose to get months on x-axis and hours on y-axis
df_normalized = df_normalized.T

# Create heatmap
fig = px.imshow(
    df_normalized,
    labels={'x': 'Month', 'y': 'Hour of Day', 'color': 'Post Frequency (%)'},
    x=df_normalized.columns,
    y=df_normalized.index,
    color_continuous_scale='Viridis',
    title='Normalized Hourly Commenting Patterns by Month',
    aspect='auto'
)

# Customize layout
fig.update_layout(
    xaxis_title='Month',
    yaxis_title=f'Hour of Day ({TZ})',
    xaxis=dict(tickangle=45, tickmode='array', tickvals=df_normalized.columns[::2]),
    yaxis=dict(tickvals=list(range(24))),
    coloraxis_colorbar=dict(title='% of Posts'),
    height=600  # Taller plot for better hour labels
)

# Add hover template
fig.update_traces(
    hovertemplate='<b>Month</b>: %{x}<br><b>Hour</b>: %{y}:00<br><b>Frequency</b>: %{z:.2f}%<extra></extra>'
)

fig.show()